# LeNet implementation
## Imports
### Libraries

In [1]:
import sys
# Uninstall existing TensorFlow and Keras to ensure a clean install.
!pip uninstall -y tensorflow keras tf-keras > /dev/null 2>&1

# Install the latest compatible version of TensorFlow.
# Using --upgrade ensures that existing packages are updated if necessary.
# Removing explicit version numbers allows pip to find the latest compatible version.
!pip install --upgrade tensorflow > /dev/null 2>&1

# Install the latest compatible version of protobuf.
!pip install --upgrade protobuf > /dev/null 2>&1

print('TensorFlow and Protobuf installations updated successfully.')

TensorFlow and Protobuf installations updated successfully.


In [2]:
import os
import numpy as np
import pandas as pd

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]: # I print 5 files per folder
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
from pathlib import Path
from skimage import io
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D, Dense, Flatten, InputLayer

### Data

In [9]:
def load_data(folder_path:str):
    path_to_images = Path(folder_path)
    ids = []
    dataset = []
    for img_file in sorted(path_to_images.iterdir()):
        id_img = "".join(img_file.name.split(".")[:-1])
        ids.append(id_img)
        img = io.imread(img_file)
        dataset.append(img)
    dataset_as_array = np.array(dataset)
    return dataset_as_array, ids

### Loading and Preprocessing MNIST Data

Since the Kaggle dataset was not found, we will use the standard MNIST dataset available through `tf.keras.datasets`. We will then preprocess it to match the expected input format for the LeNet model (28x28x1 grayscale images, normalized).

In [10]:
# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Original X_train shape: {X_train.shape}")
print(f"Original y_train shape: {y_train.shape}")
print(f"Original X_test shape: {X_test.shape}")
print(f"Original y_test shape: {y_test.shape}")

Original X_train shape: (60000, 28, 28)
Original y_train shape: (60000,)
Original X_test shape: (10000, 28, 28)
Original y_test shape: (10000,)


In [7]:
# Normalize and add channel dimension
# Images are 28x28, grayscale, so we need to add a channel dimension (28x28x1)
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

X_train_norm = np.expand_dims(X_train_norm, axis=-1)
X_test_norm = np.expand_dims(X_test_norm, axis=-1)

print(f"Normalized X_train shape: {X_train_norm.shape}")
print(f"Normalized X_test shape: {X_test_norm.shape}")

Normalized X_train shape: (60000, 28, 28, 1)
Normalized X_test shape: (10000, 28, 28, 1)


In [8]:
# One-hot encode the labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"Categorical y_train shape: {y_train_cat.shape}")
print(f"Categorical y_test shape: {y_test_cat.shape}")

Categorical y_train shape: (60000, 10)
Categorical y_test shape: (10000, 10)


## Model
### Model architecture

We want :

- Input Layer : 28 x 28 x 1
- Convolution with 5x5 kernel + 2 padding:28x28x6
- Activation : sigmoid
- Pool with 2x2 average kernel + 2 stride : 14 x 14 x 6
- Convolution with 5x5 kernel (no pad) : 10x10x16
- Activation : sigmoid
- Pool with 2x2 average kernel + 2 stride : 5x5x16
- Flatten
- Dense : 120 neurons
- Activation : sigmoid
- Dense : 84 fully connected neurons
- Activation : sigmoid
- Dense : 10 fully connected neurons
- Activation : softmax

In [18]:
model = Sequential([
    InputLayer(input_shape=(28,28,1)),

    Conv2D(
        filters= 6,
        kernel_size=(5,5),
        activation='sigmoid',
        strides=(1,1),
        padding='same',),

    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),

    Conv2D(
        filters=16,
        kernel_size=(5,5),
        strides=(1,1),
        padding='valid',
        activation='sigmoid'
    ),

    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),


    Flatten(),

    Dense(120, activation='sigmoid'),
    Dense(84, activation='sigmoid'),
    Dense(10, activation='softmax')
])

model.summary()


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Model training parameters

In [19]:
model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=["accuracy"]
)

In [20]:
history = model.fit(
    X_train_norm,
    y_train_cat,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - accuracy: 0.9031 - loss: 0.2918 - val_accuracy: 0.9728 - val_loss: 0.0904
Epoch 2/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9781 - loss: 0.0738 - val_accuracy: 0.9828 - val_loss: 0.0654
Epoch 3/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - accuracy: 0.9834 - loss: 0.0566 - val_accuracy: 0.9872 - val_loss: 0.0493
Epoch 4/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9869 - loss: 0.0453 - val_accuracy: 0.9836 - val_loss: 0.0590
Epoch 5/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9884 - loss: 0.0405 - val_accuracy: 0.9855 - val_loss: 0.0540
Epoch 6/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9899 - loss: 0.0336 - val_accuracy: 0.9848 - val_loss: 0.0582
Epoch 7/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9904 - loss: 0.0317 - val_accuracy: 0.9862 - val_loss: 0.0560
Epoch 8/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 39s 13ms/step - accuracy: 0.9917 -

In [21]:
y_pred = np.argmax(model.predict(X_test_norm), axis=1)

# Generate sequential IDs for the test set since X_test_ids are not provided by tf.keras.datasets.mnist
X_test_ids_generated = np.arange(len(X_test_norm))

submission = pd.DataFrame({
    "Id": X_test_ids_generated,
    "Label": y_pred
})
submission.to_csv("submission.csv", index=False)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
